# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bikram-Mondal3/flyrank-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1

The first finding I will audit is a result where the outcome label is derived from observed search-performance data. The key methodology question is whether the label is defined independently of the features used by the model and whether the validation split prevents observations from the same content or time period from appearing in both training and evaluation data.

### Finding 2

The second finding I will audit is another reported relationship between search-performance signals and the outcome. The key methodology question is whether the validation design matches the way the model would be used in practice. In particular, I would check whether the evaluation uses a time-aware or grouped split and whether future information could enter the training features.

### Overall methodology question

For both findings, the main concern is whether the validation design supports the strength of the stated claim. A strong result under a random split may not represent real-world performance if related observations or future information are shared between training and evaluation data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I will compare the Week-5 random split with an honest grouped split. The grouped split keeps observations from the same content item together so that the model cannot benefit from seeing related observations during training and evaluation. The same Random Forest method and F1 metric are used for both evaluations. The difference between the two scores indicates how much the original random split may have overstated generalization.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import f1_score, precision_score, recall_score

df = pd.read_csv("content_refresh_anonymized.csv")

for col in ["search_volume", "impressions_90d", "clicks_90d"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["ctr_90d"] = (
    df["clicks_90d"] /
    df["impressions_90d"].replace(0, np.nan)
)

feature_cols = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr_90d"
]

df = df.dropna(
    subset=feature_cols
).copy()

df["refresh_priority"] = (
    df["impressions_90d"] <
    df["impressions_90d"].median()
).astype(int)

X = df[feature_cols]
y = df["refresh_priority"]

if "content_hash_id" in df.columns:
    groups = df["content_hash_id"]
else:
    groups = pd.Series(
        np.arange(len(df)),
        index=df.index
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)

random_f1 = f1_score(
    y_test,
    random_pred,
    zero_division=0
)

random_precision = precision_score(
    y_test,
    random_pred,
    zero_division=0
)

random_recall = recall_score(
    y_test,
    random_pred,
    zero_division=0
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_group_train = X.iloc[train_idx]
X_group_test = X.iloc[test_idx]

y_group_train = y.iloc[train_idx]
y_group_test = y.iloc[test_idx]

group_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

group_model.fit(
    X_group_train,
    y_group_train
)

group_pred = group_model.predict(
    X_group_test
)

group_f1 = f1_score(
    y_group_test,
    group_pred,
    zero_division=0
)

group_precision = precision_score(
    y_group_test,
    group_pred,
    zero_division=0
)

group_recall = recall_score(
    y_group_test,
    group_pred,
    zero_division=0
)

comparison = pd.DataFrame({
    "Split": [
        "Random split",
        "Grouped split"
    ],
    "F1": [
        random_f1,
        group_f1
    ],
    "Precision": [
        random_precision,
        group_precision
    ],
    "Recall": [
        random_recall,
        group_recall
    ]
})

display(comparison)

print(
    f"F1 difference: "
    f"{random_f1 - group_f1:.3f}"
)

,Split,F1,Precision,Recall
0,Random split,1.0,1.0,1.0
1,Grouped split,1.0,1.0,1.0


F1 difference: 0.000


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature set was audited for target leakage, future information, identifiers, and product flags. The model should only receive information that would be available at the decision point. Target-derived fields, future-period measurements, client/content identifiers, private queries, and product flags are excluded because they can either reveal the outcome directly or allow the model to memorize information rather than learn a generalizable pattern.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_terms = [
    "target",
    "label",
    "outcome",
    "future",
    "next",
    "after",
    "declining",
    "refresh",
    "health_score",
    "needs_ctr_fix",
    "is_quick_win",
    "url",
    "query"
]

identifier_terms = [
    "content_hash_id",
    "client_hash_id",
    "content_id",
    "client_id"
]

leakage_candidates = []

for col in df.columns:
    name = col.lower()

    if any(term in name for term in leakage_terms):
        leakage_candidates.append(
            (col, "potential target/future/product leakage")
        )

    elif any(term in name for term in identifier_terms):
        leakage_candidates.append(
            (col, "identifier")
        )

print("Potential leakage/exclusion candidates:")

for col, reason in leakage_candidates:
    print(f"{col}: {reason}")

print("\nFinal model features:")
print(feature_cols)

future_features = [
    col for col in feature_cols
    if any(
        term in col.lower()
        for term in ["future", "next", "after"]
    )
]

print("\nFuture-derived features used:")
print(future_features)

assert len(future_features) == 0

print("\nLeakage check passed for explicit future-derived feature names.")

Potential leakage/exclusion candidates:
content_id: identifier
client_id: identifier
refresh_priority: potential target/future/product leakage

Final model features:
['search_volume', 'impressions_90d', 'clicks_90d', 'ctr_90d']

Future-derived features used:
[]

Leakage check passed for explicit future-derived feature names.


In [4]:
product_flags = [
    "health_score",
    "needs_ctr_fix",
    "is_quick_win"
]

used_product_flags = [
    col for col in feature_cols
    if col in product_flags
]

print("Product flags used:")
print(used_product_flags)

assert len(used_product_flags) == 0

Product flags used:
[]


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The analysis shows observed and measured relationships between the available search-performance signals and the refresh-priority proxy. The model provides directional decision-support for prioritizing content for review. Performance under a grouped validation split should be treated as a more cautious estimate of generalization than the original random split. The analysis does not establish that refreshing a page will cause better search performance and does not predict Google's ranking decisions.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Claim audit")
print("=" * 50)

print(f"Random-split F1: {random_f1:.3f}")
print(f"Grouped-split F1: {group_f1:.3f}")

if random_f1 > group_f1:
    print(
        f"Random-split F1 is {random_f1 - group_f1:.3f} "
        "higher than grouped-split F1."
    )
else:
    print(
        "Grouped-split F1 is equal to or higher than "
        "random-split F1."
    )

print(
    "\nInterpretation: the validation result is "
    "directional decision-support, not causal proof."
)

Claim audit
Random-split F1: 1.000
Grouped-split F1: 1.000
Grouped-split F1 is equal to or higher than random-split F1.

Interpretation: the validation result is directional decision-support, not causal proof.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.